# Generator Logs: Debug Statistics Analysis

This notebook reads JSON logs from `logs/` that were created by the generator's debug helper (`print_generation_summary`).

**Contents:**
- ...

**How to use:**
1. Make sure you have some JSON logs in the `logs/` folder (run the app with `DEBUG_GENERATOR=True`).
2. Run the cells below. Re-run the first two when new logs appear.

In [ ]:
# Imports
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# Settings
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

In [2]:
# Load logs/*generator_debug_*.json into a DataFrame
LOG_DIR = Path('logs')
files = sorted(LOG_DIR.glob('generator_debug_*.json'))
print(f"Found {len(files)} log file(s)")

rows = []
for p in files:
    try:
        with p.open('r', encoding='utf-8') as f:
            data = json.load(f)
        ts = data.get('timestamp')
        k = data.get('k')
        seed = data.get('seed')
        max_len = data.get('max_length')
        n_req = data.get('n_requested')
        analysis = data.get('analysis', {})
        samples = data.get('samples', [])

        # Make safe types
        ts_dt = pd.to_datetime(ts, errors='coerce')

        row = {
            'timestamp': ts_dt,
            'k': k,
            'seed': seed,
            'max_length': max_len,
            'n_requested': n_req,
            'unique_count': analysis.get('unique_count'),
            'total_count': analysis.get('total_count'),
            'avg_length': analysis.get('avg_length'),
            'hyphen_count': analysis.get('hyphen_count'),
            'underscore_count': analysis.get('underscore_count'),
            'all_start_with_seed': analysis.get('all_start_with_seed'),
            'empty_count': analysis.get('empty_count'),
            'sample_1': samples[0] if len(samples) > 0 else None,
            'sample_2': samples[1] if len(samples) > 1 else None,
            'sample_3': samples[2] if len(samples) > 2 else None,
            'file': str(p)
        }
        rows.append(row)
    except Exception as e:
        print(f"⚠️  Failed to parse {p}: {e}")

df = pd.DataFrame(rows).sort_values('timestamp')

if df.empty:
    print("No logs parsed. Run the generator with debug enabled first.")
else:
    # Derived ratios
    df['hyphen_ratio'] = df.apply(lambda r: (r.hyphen_count or 0) / r.total_count if r.total_count else 0, axis=1)
    df['unique_ratio'] = df.apply(lambda r: (r.unique_count or 0) / r.total_count if r.total_count else 0, axis=1)
    df['underscore_ratio'] = df.apply(lambda r: (r.underscore_count or 0) / r.total_count if r.total_count else 0, axis=1)
    display(df.tail(10))

Found 11 log file(s)


,timestamp,k,seed,max_length,n_requested,unique_count,total_count,avg_length,hyphen_count,underscore_count,all_start_with_seed,empty_count,sample_1,sample_2,sample_3,file,hyphen_ratio,unique_ratio,underscore_ratio
1,2025-09-21 13:14:20.347500,2,hello,10,5,5,5,9.4,1,0,True,0,helloardow,helloub,helloprinc,logs\generator_debug_20250921_131420.json,0.2,1.0,0.0
2,2025-09-21 13:14:33.098992,2,test,10,5,5,5,9.0,3,0,True,0,testass-rs,testo-ton,testelabut,logs\generator_debug_20250921_131433.json,0.6,1.0,0.0
3,2025-09-21 15:47:52.965863,2,hello,10,5,5,5,8.4,2,0,True,0,hellogrule,hellog,helloy,logs\generator_debug_20250921_154752.json,0.4,1.0,0.0
4,2025-09-21 15:48:15.595859,2,hello,10,5,5,5,9.6,3,0,True,0,hellouth,hellot-nod,helloa-pil,logs\generator_debug_20250921_154815.json,0.6,1.0,0.0
5,2025-09-21 15:51:12.300860,2,hello,10,5,1,5,5.0,0,0,True,0,hello,hello,hello,logs\generator_debug_20250921_155112.json,0.0,0.2,0.0
6,2025-09-21 16:16:54.254359,2,hello,10,5,1,5,5.0,0,0,True,0,hello,hello,hello,logs\generator_debug_20250921_161654.json,0.0,0.2,0.0
7,2025-09-21 16:17:13.385359,2,hello,10,5,1,5,5.0,0,0,True,0,hello,hello,hello,logs\generator_debug_20250921_161713.json,0.0,0.2,0.0
8,2025-09-21 16:17:35.833859,4,hello,10,5,1,5,5.0,0,0,True,0,hello,hello,hello,logs\generator_debug_20250921_161735.json,0.0,0.2,0.0
9,2025-09-21 21:36:21.438680,2,hello,10,5,5,5,9.2,1,0,True,0,helloa-kob,hellobile/,hello/stru,logs\generator_debug_20250921_213621.json,0.2,1.0,0.0
10,2025-09-21 21:39:59.420178,2,hello,10,5,5,5,9.6,1,0,True,0,helloggeng,hellocordo,hellocavel,logs\generator_debug_20250921_213959.json,0.2,1.0,0.0
